In [300]:
import numpy as np
import torch
from train_reinforce import BinPacking_Environment, get_action_from_idx
from PackingUtils import *
from ModelEMS import *
import datetime
import random
import json
import seaborn as sns
import matplotlib.pyplot as plt
import pandas
import copy

In [301]:
n_times_containers = 6
total_items = []

In [302]:
item_list = {}
container_size = np.array([(35, 23, 13), (37, 26, 13), (38, 26, 13), (40, 28, 16), (42, 30, 18), (42, 30, 40), (52, 40, 17), (54, 45, 36)] * n_times_containers)
df = pandas.read_csv("task3.csv")
# print(df)
print(df.index, df.columns)

for i in df.index:
	for t in range(df.loc[i, 'qty']):
		if item_list.get(df.loc[i, 'sta_code']) is None:
			item_list[df.loc[i, 'sta_code']] = []
		item_list[df.loc[i, 'sta_code']].append(
			{
				"size": np.array([df.loc[i, '长(CM)'], df.loc[i, '宽(CM)'], df.loc[i, '高(CM)']], dtype=np.int32),
				"sku_code": df.loc[i, 'sku_code'],
			}
		)
		total_items.append(
			{
				"size": np.array([df.loc[i, '长(CM)'], df.loc[i, '宽(CM)'], df.loc[i, '高(CM)']], dtype=np.int32),
				"sku_code": df.loc[i, 'sku_code'],
			}
		)

# print(item_list)
print(container_size)
n_items = len(item_list)
print(n_items)
orders = list(item_list.keys())

RangeIndex(start=0, stop=19585, step=1) Index(['sta_code', 'sku_code', '长(CM)', '宽(CM)', '高(CM)', 'qty'], dtype='object')
[[35 23 13]
 [37 26 13]
 [38 26 13]
 [40 28 16]
 [42 30 18]
 [42 30 40]
 [52 40 17]
 [54 45 36]
 [35 23 13]
 [37 26 13]
 [38 26 13]
 [40 28 16]
 [42 30 18]
 [42 30 40]
 [52 40 17]
 [54 45 36]
 [35 23 13]
 [37 26 13]
 [38 26 13]
 [40 28 16]
 [42 30 18]
 [42 30 40]
 [52 40 17]
 [54 45 36]
 [35 23 13]
 [37 26 13]
 [38 26 13]
 [40 28 16]
 [42 30 18]
 [42 30 40]
 [52 40 17]
 [54 45 36]
 [35 23 13]
 [37 26 13]
 [38 26 13]
 [40 28 16]
 [42 30 18]
 [42 30 40]
 [52 40 17]
 [54 45 36]
 [35 23 13]
 [37 26 13]
 [38 26 13]
 [40 28 16]
 [42 30 18]
 [42 30 40]
 [52 40 17]
 [54 45 36]]
6847


In [303]:
print(list(item_list.keys())[0])
print(item_list["ODO1694009215440871"])

ODO1694009215440871
[{'size': array([28, 20, 12], dtype=int32), 'sku_code': 'SX4863-960'}, {'size': array([34, 21, 12], dtype=int32), 'sku_code': 'CW2288-001'}, {'size': array([28, 20, 12], dtype=int32), 'sku_code': 'SX4863-960'}, {'size': array([28, 20, 12], dtype=int32), 'sku_code': 'SX4863-960'}]


In [304]:
clip_num = 100
device = torch.device("cpu")
class ContainerState:
	def __init__(self, container_size):
		self.container_size = container_size
		self.height_map = np.zeros(container_size[:2])
		self.total_volume = np.prod(container_size)
		self.packed_volume = 0
		self.assigned_items = []
	
	def add_item(self, item, position, orientation):
		if not self.is_valid(item, [position[0], position[1], orientation]):
			return False
		# print(item, position, orientation, self.container_size)
		new_height_map = update_height_map(
			self.height_map,
			item,
			position,
			orientation
		)
		self.packed_volume += np.prod(item)
		self.height_map = new_height_map
		return True
	
	def get_next_state(self, item, return_type='torch'):
		placement, mask = generate_EMS_and_mask(
			height_map=self.height_map, 
			coming_item=item,
			height_limit=self.container_size[2],
			clip_num=clip_num
		)
		item_state = np.array([
			item, 
			[item[1], item[0], item[2]]
		])
		# print("ENV: ", placement.shape)
		if return_type == 'torch':
			height_map = torch.Tensor(self.height_map).reshape(1, 1, self.container_size[0], self.container_size[1]).to(device)
			placement = torch.Tensor(placement).reshape(1, -1, 5).to(device)
			item_state = torch.Tensor(item_state).reshape(1, 2, 3).to(device)
			mask = torch.Tensor(mask).reshape(1, 2, -1).to(device)
			return placement, item_state, height_map, mask
		elif return_type == 'numpy':
			return placement, item_state, self.height_map, mask
		else:
			raise ValueError("Invalid return type")
	
	def is_full(self):
		return self.packed_volume >= self.total_volume
	
	def is_valid(self, item, action):
		li, wi, hi = item.tolist()
		x, y, r = action
		if r == 1:
			li, wi = wi, li
		# print(li, wi, hi, action)
		if x-li+1 < 0 or y+wi > self.container_size[1]:
			return False
		stable_cnt = positionally_stable(
			self.height_map[x-li+1:x+1, y:y+wi],
			li, wi, hi, self.container_size[2]
		)
		if stable_cnt > 0:
			return True
		return False
	
	def get_valid_actions(self, item_list):
		valid_actions = []
		for item in item_list:
			placement, item_state, height_map, mask = self.get_next_state(item, return_type='numpy')
			if placement is not None:
				for t in range(clip_num):
					position = placement[t, :2].astype(np.int32).tolist()
					if self.is_valid(item, position + [0]):
						valid_actions.append({"item": item, "position": position, "rotation": 0})
					if self.is_valid(item, position + [1]):
						valid_actions.append({"item": item, "position": position, "rotation": 1})
		return valid_actions
	
	def remaining_capacity(self):
		return self.total_volume - self.packed_volume

cs = ContainerState((10, 10, 10))
cs.get_valid_actions([np.array([2, 3, 4]), np.array([9, 3, 2])])

[{'item': array([2, 3, 4]), 'position': [1, 0], 'rotation': 0},
 {'item': array([2, 3, 4]), 'position': [9, 0], 'rotation': 0},
 {'item': array([2, 3, 4]), 'position': [9, 0], 'rotation': 1},
 {'item': array([2, 3, 4]), 'position': [1, 7], 'rotation': 0},
 {'item': array([2, 3, 4]), 'position': [9, 7], 'rotation': 0},
 {'item': array([2, 3, 4]), 'position': [9, 7], 'rotation': 1},
 {'item': array([2, 3, 4]), 'position': [2, 0], 'rotation': 0},
 {'item': array([2, 3, 4]), 'position': [2, 0], 'rotation': 1},
 {'item': array([2, 3, 4]), 'position': [9, 0], 'rotation': 0},
 {'item': array([2, 3, 4]), 'position': [9, 0], 'rotation': 1},
 {'item': array([2, 3, 4]), 'position': [2, 8], 'rotation': 1},
 {'item': array([2, 3, 4]), 'position': [9, 8], 'rotation': 1},
 {'item': array([9, 3, 2]), 'position': [8, 0], 'rotation': 0},
 {'item': array([9, 3, 2]), 'position': [8, 0], 'rotation': 1},
 {'item': array([9, 3, 2]), 'position': [9, 0], 'rotation': 0},
 {'item': array([9, 3, 2]), 'position': 

In [305]:
model_filename="models/model_20250112_124243_.pth"
model = BPP_Model_EMS(num_placement=clip_num, batch_size=1, embed_size=128, feature_mlp_layers=[256, 128], feature_mlp_output=64).to(device)
model.load_state_dict(torch.load(model_filename, map_location=device, weights_only=True))

class MultiContainerState:
	def __init__(self, container_size, item_list):
		n_containers = container_size.shape[0]
		self.container_size = container_size
		self.remaining_items = copy.deepcopy(item_list)
		self.containers = [ContainerState(container_size[i, :]) for i in range(n_containers)]

	def is_terminal(self):
		return len(self.remaining_items) == 0  # Terminal if all items are packed

	def clone(self):
		new_containers = [container.clone() for container in self.containers]
		return MultiContainerState(new_containers, self.remaining_items[:])

In [306]:
class TreeNode:
	def __init__(self, state: 'BinPacking_Environment', parent=None, action_idx=None, reward=0):
		self.state = state  # 当前环境状态
		self.parent = parent  # 父节点
		self.action_idx = action_idx  # 从父节点到当前节点的动作索引
		self.reward = reward  # 当前动作的奖励
		self.children = []  # 子节点列表

	def is_leaf(self):
		return len(self.children) == 0

	def add_child(self, child_node):
		self.children.append(child_node)

In [307]:
def assign_items(state, container_id):
	new_state = copy.deepcopy(state)
	selected_container = new_state.containers[container_id]
	# print(new_state, new_state.containers, selected_container)
	remaining_size = selected_container.remaining_capacity()

	remaining_items = []
	for item in new_state.remaining_items:
		item_volume = np.prod(item["size"])
		if item_volume <= remaining_size:
			if random.random() < 0.5:
				selected_container.assigned_items.append(item)
				remaining_size -= item_volume
			else:
				remaining_items.append(item)
		else:
			remaining_items.append(item)

	# Remove assigned items from remaining items
	# for item in new_state.remaining_items:
	# new_state.remaining_items = [
	# 	item for item in new_state.remaining_items if item not in selected_container.assigned_items
	# ]
	new_state.remaining_items = remaining_items

	return new_state

def pack_items(state, container_id, return_actions=False):
	new_state = copy.deepcopy(state)
	selected_container = new_state.containers[container_id]
	actions_list = []
	for item_info in selected_container.assigned_items:
		item = item_info["size"]
		# item = item_info
		pack_states = selected_container.get_next_state(item)
		if torch.sum(pack_states[-1]) == 0:
			return state, False
		ems_state, item_state, ems_feature, item_feature, logits_raw_, logits = model(*pack_states)
		action_idx = torch.argmax(logits, dim=-1).item()
		action = get_action_from_idx(pack_states[0], action_idx, clip_num)
		# print(action)
		if not selected_container.add_item(item, action[:2], action[2]):
			return state, False
		actions_list.append((item, action))
	if not return_actions:
		return new_state, True
	else:
		return new_state, True, actions_list

In [308]:
def dfs_search(state, depth=0, max_depth=100):
	"""
	DFS algorithm with a single-container policy model for packing.

	Args:
		state (MultiContainerState): Current state of the containers and items.
		model: Policy model for single-container packing.
		depth (int): Current depth of the search.
		max_depth (int): Maximum depth to prevent infinite recursion.

	Returns:
		MultiContainerState: The final packed state, or None if no solution exists.
	"""
	if state.is_terminal():
		return state  # All items are packed

	if depth >= max_depth:
		return None  # Prevent infinite recursion
	print("depth = {}".format(depth))
	for container_id, container in enumerate(state.containers):
		# Assign items to the container
		assigned_state = assign_items(state, container_id)

		# Pack the assigned items into the container
		packed_container, succeed = pack_items(assigned_state, container_id)
		if not succeed:
			continue
		# assigned_state = packed_container

		# Recursively search
		result = dfs_search(packed_container, depth + 1, max_depth)
		if result:
			return result  # Return the first valid solution
		assigned_state.containers[container_id].assigned_items = []

	return None  # No solution found

In [309]:
# item_selected = item_list["ODO1694009215440871"]
# print(orders)
item_selected = []
n_choices = 6
choices = np.random.choice(orders, n_choices, replace=False)
print(choices)
for choice in choices:
	item_selected.extend(item_list[choice])
print(item_selected)

['TBNK2309332347' 'ODO1693866742016696' 'TBBN2309071641' 'TBNK2309147766'
 'ODO1694268170092687' 'BSIN2309020884']
[{'size': array([35, 24, 13], dtype=int32), 'sku_code': 'DJ3624-002'}, {'size': array([35, 24, 13], dtype=int32), 'sku_code': 'DJ3624-002'}, {'size': array([33, 30,  7], dtype=int32), 'sku_code': 'SX7677-010'}, {'size': array([35, 23, 12], dtype=int32), 'sku_code': 'DZ5485-410'}, {'size': array([30, 23,  5], dtype=int32), 'sku_code': 'DH4058-011'}, {'size': array([35, 23, 12], dtype=int32), 'sku_code': 'DZ5485-410'}, {'size': array([27, 22, 11], dtype=int32), 'sku_code': 'FJ7689-101'}, {'size': array([23,  5,  1], dtype=int32), 'sku_code': 'AC2286-010'}, {'size': array([38, 30,  1], dtype=int32), 'sku_code': 'DO7393-010'}, {'size': array([33, 20, 13], dtype=int32), 'sku_code': 'FJ7687-101'}, {'size': array([33, 20, 13], dtype=int32), 'sku_code': 'FJ7687-101'}, {'size': array([33, 20, 13], dtype=int32), 'sku_code': 'FJ7687-101'}, {'size': array([33, 20, 13], dtype=int32), '

In [310]:
initial_state = MultiContainerState(container_size=container_size, item_list=item_selected)
result = dfs_search(initial_state, )


depth = 0
depth = 1
depth = 2
depth = 3
depth = 4
depth = 3
depth = 4
depth = 5
depth = 6
depth = 7
depth = 8
depth = 9
depth = 10
depth = 11
depth = 12
depth = 13
depth = 14
depth = 15
depth = 16


In [311]:
print(result)

In [312]:
def get_containers_report(state):
	for id, container in enumerate(state.containers):
		if container.packed_volume == 0:
			continue
		print(f"Container {id}: {container.container_size}, items: {container.assigned_items}")
		# print(f"height_map: {container.height_map}")

In [313]:
get_containers_report(result)

Container 0: [35 23 13], items: [{'size': array([35, 23, 12], dtype=int32), 'sku_code': 'DZ5485-410'}]
Container 1: [37 26 13], items: [{'size': array([27, 22, 11], dtype=int32), 'sku_code': 'FJ7689-101'}]
Container 2: [38 26 13], items: [{'size': array([34, 24, 12], dtype=int32), 'sku_code': 'DM3493-002'}]
Container 4: [42 30 18], items: [{'size': array([33, 30,  7], dtype=int32), 'sku_code': 'SX7677-010'}]
Container 5: [42 30 40], items: [{'size': array([23,  5,  1], dtype=int32), 'sku_code': 'AC2286-010'}, {'size': array([34, 24, 12], dtype=int32), 'sku_code': 'DM3493-002'}, {'size': array([28, 22, 11], dtype=int32), 'sku_code': 'FN3687-181'}, {'size': array([28, 22, 11], dtype=int32), 'sku_code': 'FN3687-181'}]
Container 7: [54 45 36], items: [{'size': array([33, 20, 13], dtype=int32), 'sku_code': 'FJ7687-101'}, {'size': array([33, 20, 13], dtype=int32), 'sku_code': 'FJ7687-101'}, {'size': array([38, 29,  2], dtype=int32), 'sku_code': 'BV2667-063'}, {'size': array([28, 22, 11], dty